In [1]:
# install the dependencies (pandas, numpy, openpyxl for .xlsx, ...)
%pip install pandas numpy openpyxl matplotlib seaborn scikit-learn -q

# import the dependencies
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Rolling sales by borough: one Excel file per NYC borough under ./data
DATA_DIR = Path("data")

# (display name, filename) — values must be borough labels, not paths alone
BOROUGH_FILES = [
    ("Bronx", "rollingsales_bronx.xlsx"),
    ("Brooklyn", "rollingsales_brooklyn.xlsx"),
    ("Manhattan", "rollingsales_manhattan.xlsx"),
    ("Queens", "rollingsales_queens.xlsx"),
    ("Staten Island", "rollingsales_statenisland.xlsx"),
]

borough_frames: dict[str, pd.DataFrame] = {}
for borough, fname in BOROUGH_FILES:
    path = DATA_DIR / fname
    # First 4 rows are title/description; real table header starts on row 5
    df = pd.read_excel(path, skiprows=4)

    print(f"Shape of {borough} data: {df.shape}")

    borough_frames[borough] = df

# Remove any rows where the sale price is 0 or negative or missing
for borough in borough_frames:
    borough_frames[borough] = borough_frames[borough][borough_frames[borough]["SALE PRICE"].notna()]
    borough_frames[borough] = borough_frames[borough][borough_frames[borough]["SALE PRICE"] > 0]

sales = pd.concat(borough_frames.values(), ignore_index=True)

sales.shape, list(borough_frames.keys())

Shape of Bronx data: (6498, 21)
Shape of Brooklyn data: (22641, 21)
Shape of Manhattan data: (19163, 21)
Shape of Queens data: (26385, 21)
Shape of Staten Island data: (7036, 21)


((53956, 21), ['Bronx', 'Brooklyn', 'Manhattan', 'Queens', 'Staten Island'])

In [3]:
# Dropping irrelevant columns
columns_to_drop = ["BOROUGH", "APARTMENT NUMBER", "EASEMENT"]
for borough in borough_frames:
    try:
        borough_frames[borough] = borough_frames[borough].drop(columns=columns_to_drop)
    except KeyError:
        pass  # column already absent in this frame

In [4]:


# How empty is each column in each borough?
def is_nullish(s: pd.Series) -> pd.Series:
    null = s.isna()
    if s.dtype != object and not pd.api.types.is_string_dtype(s):
        return null
    empty_str = s.map(lambda x: isinstance(x, str) and x.strip() == "")
    return null | empty_str

# Fill in missing values for residential, commercial, and total units
for borough in borough_frames:
    df = borough_frames[borough]
    for idx, row in df.iterrows():
        residential_units = row["RESIDENTIAL UNITS"]
        commercial_units = row["COMMERCIAL UNITS"]
        total_units = row["TOTAL UNITS"]

        r_ok = pd.notna(residential_units) and not (
            isinstance(residential_units, str) and residential_units.strip() == ""
        )
        c_ok = pd.notna(commercial_units) and not (
            isinstance(commercial_units, str) and commercial_units.strip() == ""
        )
        t_ok = pd.notna(total_units) and not (
            isinstance(total_units, str) and total_units.strip() == ""
        )

        if not r_ok and t_ok and c_ok:
            residential_units = total_units - commercial_units
        elif not t_ok and r_ok and c_ok:
            total_units = residential_units + commercial_units
        elif not c_ok and t_ok and r_ok:
            commercial_units = total_units - residential_units

        df.loc[idx, "RESIDENTIAL UNITS"] = residential_units
        df.loc[idx, "COMMERCIAL UNITS"] = commercial_units
        df.loc[idx, "TOTAL UNITS"] = total_units

# Rebuild combined frame so later analysis sees imputed unit columns
sales = pd.concat(borough_frames.values(), ignore_index=True)

for borough in borough_frames:
    print(f"Percentage of empty values in {borough} data:")
    for col in borough_frames[borough].columns:
        pct = is_nullish(borough_frames[borough][col]).mean()
        print(f"{col}: {pct}")
    print()

Percentage of empty values in Bronx data:
NEIGHBORHOOD: 0.0
BUILDING CLASS CATEGORY: 0.0
TAX CLASS AT PRESENT: 0.0
BLOCK: 0.0
LOT: 0.0
BUILDING CLASS AT PRESENT: 0.0
ADDRESS: 0.0
ZIP CODE: 0.0
RESIDENTIAL UNITS: 0.22034296452901103
COMMERCIAL UNITS: 0.22034296452901103
TOTAL UNITS: 0.22034296452901103
LAND SQUARE FEET: 0.30913789053323937
GROSS SQUARE FEET: 0.30913789053323937
YEAR BUILT: 0.09701667841202725
TAX CLASS AT TIME OF SALE: 0.0
BUILDING CLASS AT TIME OF SALE: 0.0
SALE PRICE: 0.0
SALE DATE: 0.0

Percentage of empty values in Brooklyn data:
NEIGHBORHOOD: 0.0
BUILDING CLASS CATEGORY: 0.0
TAX CLASS AT PRESENT: 0.0
BLOCK: 0.0
LOT: 0.0
BUILDING CLASS AT PRESENT: 0.0
ADDRESS: 0.0
ZIP CODE: 7.126060001425212e-05
RESIDENTIAL UNITS: 0.16183282263236656
COMMERCIAL UNITS: 0.16183282263236656
TOTAL UNITS: 0.16183282263236656
LAND SQUARE FEET: 0.45934582769186916
GROSS SQUARE FEET: 0.45934582769186916
YEAR BUILT: 0.06719874581343975
TAX CLASS AT TIME OF SALE: 0.0
BUILDING CLASS AT TIME OF

In [5]:
# Work with manhattan coops (exclude categories that also contain CONDO, e.g. CONDO COOPS)
_bcc = borough_frames["Manhattan"]["BUILDING CLASS CATEGORY"].astype(str).str.upper()
manhattan_condos = borough_frames["Manhattan"][_bcc.str.contains("CONDO") & ~_bcc.str.contains("COOP")]

# Total number of rentals
print("Total number of rentals:")
print(manhattan_condos.shape[0])

# See what different Building class categories are there
print("Building class categories:")
print(manhattan_condos["BUILDING CLASS CATEGORY"].unique())

# see the columns one more time
print(manhattan_condos.columns)

Total number of rentals:
6918
Building class categories:
<StringArray>
[             '11 SPECIAL CONDO BILLING LOTS',
              '12 CONDOS - WALKUP APARTMENTS',
            '13 CONDOS - ELEVATOR APARTMENTS',
          '15 CONDOS - 2-10 UNIT RESIDENTIAL',
                   '46 CONDO STORE BUILDINGS',
 '16 CONDOS - 2-10 UNIT WITH COMMERCIAL UNIT',
                       '28 COMMERCIAL CONDOS',
                  '43 CONDO OFFICE BUILDINGS',
                           '44 CONDO PARKING',
              '47 CONDO NON-BUSINESS STORAGE',
          '48 CONDO TERRACES/GARDENS/CABANAS',
  '42 CONDO CULTURAL/MEDICAL/EDUCATIONAL/ETC',
                            '45 CONDO HOTELS',
                      '04 TAX CLASS 1 CONDOS',
          '49 CONDO WAREHOUSES/FACTORY/INDUS']
Length: 15, dtype: str
Index(['NEIGHBORHOOD', 'BUILDING CLASS CATEGORY', 'TAX CLASS AT PRESENT',
       'BLOCK', 'LOT', 'BUILDING CLASS AT PRESENT', 'ADDRESS', 'ZIP CODE',
       'RESIDENTIAL UNITS', 'COMMERCIAL UNITS', 'TOT

In [6]:
# print the head
print(manhattan_condos.head())

     NEIGHBORHOOD          BUILDING CLASS CATEGORY TAX CLASS AT PRESENT  \
71  ALPHABET CITY    11 SPECIAL CONDO BILLING LOTS                    2   
72  ALPHABET CITY    12 CONDOS - WALKUP APARTMENTS                    2   
74  ALPHABET CITY    12 CONDOS - WALKUP APARTMENTS                    2   
78  ALPHABET CITY  13 CONDOS - ELEVATOR APARTMENTS                    2   
79  ALPHABET CITY  13 CONDOS - ELEVATOR APARTMENTS                    2   

    BLOCK   LOT BUILDING CLASS AT PRESENT                      ADDRESS  \
71    397  1301                        RR   250 EAST HOUSTON STREET, 1   
72    394  1203                        R2     615 EAST 11TH STREET, B1   
74    405  1207                        R2       511 EAST 11 STREET, A3   
78    384  1215                        R4  310 EAST HOUSTON STREET, 5A   
79    392  1022                        R4             143 AVENUE B, 6A   

    ZIP CODE  RESIDENTIAL UNITS  COMMERCIAL UNITS  TOTAL UNITS  \
71     10002              132.0       

In [7]:
# Print the columsn that are entirely null
print(manhattan_condos.columns[manhattan_condos.isna().all()])

# remove any columsn that are entirely null
manhattan_condos = manhattan_condos.dropna(axis=1, how="all")

# print the head
print(manhattan_condos.head())

Index([], dtype='str')
     NEIGHBORHOOD          BUILDING CLASS CATEGORY TAX CLASS AT PRESENT  \
71  ALPHABET CITY    11 SPECIAL CONDO BILLING LOTS                    2   
72  ALPHABET CITY    12 CONDOS - WALKUP APARTMENTS                    2   
74  ALPHABET CITY    12 CONDOS - WALKUP APARTMENTS                    2   
78  ALPHABET CITY  13 CONDOS - ELEVATOR APARTMENTS                    2   
79  ALPHABET CITY  13 CONDOS - ELEVATOR APARTMENTS                    2   

    BLOCK   LOT BUILDING CLASS AT PRESENT                      ADDRESS  \
71    397  1301                        RR   250 EAST HOUSTON STREET, 1   
72    394  1203                        R2     615 EAST 11TH STREET, B1   
74    405  1207                        R2       511 EAST 11 STREET, A3   
78    384  1215                        R4  310 EAST HOUSTON STREET, 5A   
79    392  1022                        R4             143 AVENUE B, 6A   

    ZIP CODE  RESIDENTIAL UNITS  COMMERCIAL UNITS  TOTAL UNITS  \
71     10002   

In [8]:
# print out what percent of each column is null
for col in manhattan_condos.columns:
    print(f"{col}: {manhattan_condos[col].isna().mean()}")


NEIGHBORHOOD: 0.0
BUILDING CLASS CATEGORY: 0.0
TAX CLASS AT PRESENT: 0.0
BLOCK: 0.0
LOT: 0.0
BUILDING CLASS AT PRESENT: 0.0
ADDRESS: 0.0
ZIP CODE: 0.0
RESIDENTIAL UNITS: 0.0
COMMERCIAL UNITS: 0.0
TOTAL UNITS: 0.0
LAND SQUARE FEET: 0.9978317432784042
GROSS SQUARE FEET: 0.9978317432784042
YEAR BUILT: 0.1688349233882625
TAX CLASS AT TIME OF SALE: 0.0
BUILDING CLASS AT TIME OF SALE: 0.0
SALE PRICE: 0.0
SALE DATE: 0.0
